# Entendimento inicial dos dados

Esta etapa responde o que realmente existe nos arquivos do SECOM antes de qualquer tratamento. Vamos verificar tamanho, estrutura, registros, rótulos, timestamps e valores ausentes.

Nenhuma coluna será removida, renomeada ou transformada neste notebook. As conclusões serão baseadas apenas no que for observado nos arquivos e na documentação oficial.

In [3]:
from pathlib import Path
import re

import pandas as pd


# Define os caminhos a partir da raiz do projeto, independentemente do diretório atual do notebook.
raiz_projeto = Path.cwd().parent
pasta_dados = raiz_projeto / "data" / "raw"
caminho_dados = pasta_dados / "secom.data"
caminho_rotulos = pasta_dados / "secom_labels.data"
caminho_documentacao = pasta_dados / "secom.names"

print(f"Raiz do projeto: {raiz_projeto}")
print(f"Pasta de dados: {pasta_dados}")

Raiz do projeto: c:\Users\kaire\OneDrive\Área de Trabalho\Estudos\Industrial Efficiency & Quality Analytics
Pasta de dados: c:\Users\kaire\OneDrive\Área de Trabalho\Estudos\Industrial Efficiency & Quality Analytics\data\raw


## 1. Tamanho e estrutura física

Os arquivos foram medidos antes da leitura. O maior arquivo tem aproximadamente 5,4 MB, portanto a leitura controlada da base completa é aceitável neste ambiente. Para bases maiores, usaríamos amostragem, chunks ou leitura seletiva de colunas.

In [6]:
# Lista tamanho, quantidade de bytes e extensão dos arquivos brutos.
arquivos_brutos = []
for caminho in sorted(pasta_dados.iterdir()):
    if caminho.is_file():
        arquivos_brutos.append(
            {
                "arquivo": caminho.name,
                "tamanho_bytes": caminho.stat().st_size,
                "tamanho_mb": round(caminho.stat().st_size / (1024**2), 3),
            }
        )

pd.DataFrame(arquivos_brutos)

,arquivo,tamanho_bytes,tamanho_mb
0,README.md,1981,0.002
1,secom.data,5389983,5.140
2,secom.names,4223,0.004
3,secom.zip,1964989,1.874
4,secom_labels.data,40638,0.039


## 2. Documentação fornecida

A documentação é consultada como evidência, mas será comparada com os arquivos reais. Não vamos assumir que a descrição publicada corresponde perfeitamente ao conteúdo físico sem verificar.

In [7]:
# Exibe as primeiras linhas da documentação oficial incluída no download.
documentacao = caminho_documentacao.read_text(encoding="utf-8", errors="replace")
print("\n".join(documentacao.splitlines()[:40]))

Title: SECOM Data Set

Abstract: Data from a semi-conductor manufacturing process
	

-----------------------------------------------------

Data Set Characteristics: Multivariate
Number of Instances: 1567
Area: Computer
Attribute Characteristics: Real
Number of Attributes: 591
Date Donated: 2008-11-19
Associated Tasks: Classification, Causal-Discovery
Missing Values? Yes

-----------------------------------------------------

Source:

Authors: Michael McCann, Adrian Johnston 

-----------------------------------------------------

Data Set Information:

A complex modern semi-conductor manufacturing process is normally under consistent 
surveillance via the monitoring of signals/variables collected from sensors and or 
process measurement points. However, not all of these signals are equally valuable 
in a specific monitoring system. The measured signals contain a combination of 
useful information, irrelevant information as well as noise. It is often the case 
that useful information i

## 3. Leitura dos arquivos

O arquivo de medições não possui cabeçalho e usa espaços como separadores. Os nomes das colunas ainda não serão inventados.

O arquivo de rótulos possui duas informações por linha: o resultado do teste e o timestamp entre aspas. O parsing é feito explicitamente para preservar a data e hora como uma única informação.

In [8]:
# Lê as medições sem atribuir nomes semânticos às variáveis anonimizadas.
dados = pd.read_csv(
    caminho_dados,
    sep=r"\s+",
    header=None,
    na_values="NaN",
    engine="python",
)

# Extrai rótulo e timestamp com uma expressão explícita para preservar a data entre aspas.
padrao_rotulo = re.compile(r'^\s*(-?\d+)\s+"([^"]+)"\s*$')
linhas_rotulos = caminho_rotulos.read_text(encoding="utf-8").splitlines()
rotulos_extraidos = [
    padrao_rotulo.match(linha).groups()
    for linha in linhas_rotulos
    if padrao_rotulo.match(linha)
]
rotulos = pd.DataFrame(rotulos_extraidos, columns=["rotulo", "timestamp"])
rotulos["rotulo"] = rotulos["rotulo"].astype(int)
rotulos["timestamp"] = pd.to_datetime(
    rotulos["timestamp"],
    format="%d/%m/%Y %H:%M:%S",
)

print(f"Formato das medições: {dados.shape}")
print(f"Formato dos rótulos: {rotulos.shape}")
dados.head(3)

Formato das medições: (1567, 590)
Formato dos rótulos: (1567, 2)


,0,1,2,3,4,5,6,7,8,9,...,580,581,582,583,584,585,586,587,588,589
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,NaN,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,0.0060,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,0.0148,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602


## 4. Primeiras verificações

Estas verificações descrevem a base. A diferença entre a quantidade documentada de features e a quantidade lida será investigada antes de qualquer tratamento.

In [9]:
# Resume dimensões, tipos, ausências e valores distintos do resultado.
resumo_inicial = pd.DataFrame(
    {
        "item": [
            "quantidade de observações nas medições",
            "quantidade de variáveis nas medições",
            "quantidade de observações nos rótulos",
            "quantidade total de valores ausentes",
            "valores distintos do rótulo",
            "primeiro timestamp",
            "último timestamp",
        ],
        "valor": [
            dados.shape[0],
            dados.shape[1],
            rotulos.shape[0],
            int(dados.isna().sum().sum()),
            sorted(rotulos["rotulo"].unique().tolist()),
            rotulos["timestamp"].min(),
            rotulos["timestamp"].max(),
        ],
    }
)
resumo_inicial

,item,valor
0,quantidade de observações nas medições,1567
1,quantidade de variáveis nas medições,590
2,quantidade de observações nos rótulos,1567
3,quantidade total de valores ausentes,41951
4,valores distintos do rótulo,"[-1, 1]"
5,primeiro timestamp,2008-07-19 11:55:00
6,último timestamp,2008-10-17 06:07:00


In [10]:
# Verifica se as linhas de medições e rótulos podem ser associadas pela posição.
validacao_alinhamento = {
    "mesma_quantidade_de_linhas": len(dados) == len(rotulos),
    "tipos_dos_rotulos": rotulos["rotulo"].value_counts(dropna=False).to_dict(),
    "timestamps_invalidos": int(rotulos["timestamp"].isna().sum()),
    "variaveis_com_ausencia": int((dados.isna().sum() > 0).sum()),
}
validacao_alinhamento

{'mesma_quantidade_de_linhas': True,
 'tipos_dos_rotulos': {-1: 1463, 1: 104},
 'timestamps_invalidos': 0,
 'variaveis_com_ausencia': 538}

## Conclusão provisória

Neste ponto, registramos apenas a estrutura observada. A próxima investigação será entender a divergência entre a documentação do UCI e a quantidade de valores por linha, além de avaliar se existe alguma informação estrutural nos arquivos que explique essa diferença. Nenhuma decisão de tratamento será tomada com base apenas nessa primeira leitura.

## 5. Investigação da divergência documental

A documentação oficial do UCI informa 591 atributos, mas a leitura física do arquivo confirma 590 valores por linha, sem cabeçalho e sem coluna de identificação. Essa diferença precisa ser registrada como limitação de origem antes de qualquer inferência analítica.

A próxima etapa é confirmar se o desvio é estrutural e não apenas um erro de leitura, e se há alguma evidência nos arquivos de suporte para decidir como tratar a base.
## Verificação final da discrepância de atributos

# Confirma o total documentado contra o total efetivamente lido.
features_documentadas = 591
features_lidas = dados.shape[1]
linhas = dados.shape[0]

resultado_diferenca = {
    "features_documentadas": features_documentadas,
    "features_lidas_no_arquivo": features_lidas,
    "diferenca": features_documentadas - features_lidas,
    "linhas_no_arquivo": linhas,
    "todas_as_linhas_com_590_tokens": bool((dados.notna().sum(axis=1) == features_lidas).all()),
}

resultado_diferenca

## 6. Diagnóstico de qualidade

Agora que a estrutura da base foi confirmada, avaliamos a qualidade dos dados antes de qualquer transformação. Esse passo é essencial porque o SECOM contém muitas variáveis com valores ausentes e isso pode distorcer análises, estatísticas e modelos, se não for tratado com cuidado.

Termos importantes:
- `missing`: valor ausente; em português, "faltante" ou "ausente".
- `NaN`: sigla de "Not a Number"; é o formato usado pelo pandas para representar valores ausentes numéricos.
- `null`: termo em inglês para "nulo" ou "ausente".
- `duplicate`: linha duplicada, isto é, observação repetida.
- `constant column`: coluna constante, quando quase todos os valores são iguais.
- `outlier`: valor extremo, que se afasta muito do restante da distribuição.
- `label`: rótulo, ou variável alvo. No caso do SECOM, representa a qualidade do processo.
- `feature`: característica ou variável explicativa, isto é, uma coluna do dataset.

A ideia aqui não é remover dados imediatamente. O objetivo é diagnosticar o cenário real para decidir depois qual tratamento é justificável.

In [11]:
# Diagnóstico de qualidade inicial do dataset
# Muitos desses indicadores ajudam a decidir se a base está pronta para análise ou precisa de tratamento.

diagnostico_qualidade = pd.DataFrame(
    {
        "métrica": [
            "quantidade de observações",
            "quantidade de variáveis",
            "linhas duplicadas",
            "valores ausentes no total",
            "linhas com pelo menos um missing",
            "colunas com pelo menos um missing",
            "colunas constantes",
            "menor rótulo observado",
            "maior rótulo observado",
            "percentual de rótulos -1",
            "percentual de rótulos 1",
        ],
        "valor": [
            int(dados.shape[0]),
            int(dados.shape[1]),
            int(dados.duplicated().sum()),
            int(dados.isna().sum().sum()),
            int(dados.isna().any(axis=1).sum()),
            int((dados.isna().sum() > 0).sum()),
            int((dados.nunique(dropna=True) <= 1).sum()),
            int(rotulos["rotulo"].min()),
            int(rotulos["rotulo"].max()),
            round(rotulos["rotulo"].value_counts(normalize=True).get(-1, 0) * 100, 2),
            round(rotulos["rotulo"].value_counts(normalize=True).get(1, 0) * 100, 2),
        ],
    }
)

diagnostico_qualidade

# As colunas com maior proporção de missing ajudam a identificar os pontos mais frágeis da base.
missing_por_coluna = (
    dados.isna().mean().sort_values(ascending=False).head(10) * 100
).round(2).rename("percentual_missing")
missing_por_coluna

# Frequência dos rótulos ajuda a entender se a variável alvo está balanceada ou desbalanceada.
rotulos_resumo = pd.DataFrame(
    {
        "contagem": rotulos["rotulo"].value_counts().sort_index(),
        "percentual": (rotulos["rotulo"].value_counts(normalize=True).sort_index() * 100).round(2),
    }
)
rotulos_resumo

,contagem,percentual
rotulo,,
-1,1463,93.36
1,104,6.64


## 7. Interpretação da variável alvo

A documentação oficial do dataset informa que o rótulo foi codificado assim:

- `-1` = passou no teste / aprovação
- `1` = falhou no teste / defeito

Esse detalhe é importante porque a variável alvo tem sinal invertido em relação ao que muitas pessoas esperam ao ver um valor positivo. Em linguagem prática, o entendimento correto do problema é: o valor `1` indica falha de qualidade e o valor `-1` indica conformidade.

Em análise, isso costuma ser convertida para uma variável booleana mais intuitiva, como `falha = 1` para defeito e `falha = 0` para aprovação.

In [12]:
# Codifica a variável alvo de forma mais legível para a análise.
# -1 = pass, 1 = fail
rotulos["status"] = rotulos["rotulo"].map({-1: "pass", 1: "fail"})
rotulos["falha"] = (rotulos["rotulo"] == 1).astype(int)

rotulos[["rotulo", "status", "falha"]].head(10)

# Resumo da distribuição da variável alvo em termos mais intuitivos.
rotulos_resumo_final = pd.DataFrame(
    {
        "status": ["pass", "fail"],
        "contagem": [
            int((rotulos["falha"] == 0).sum()),
            int((rotulos["falha"] == 1).sum()),
        ],
        "percentual": [
            round((rotulos["falha"] == 0).mean() * 100, 2),
            round((rotulos["falha"] == 1).mean() * 100, 2),
        ],
    }
)
rotulos_resumo_final

,status,contagem,percentual
0,pass,1463,93.36
1,fail,104,6.64
